# OKX Historical Data Pipeline

Downloads historical market data from the OKX exchange for a perpetual futures carry / cross-exchange funding arbitrage strategy.

**Data collected:**

| Data Type | Source | Coverage | Auth Required |
|-----------|--------|----------|---------------|
| Funding rates (perp) | CDN bulk + REST API | Mar 2022 - present | No |
| Order book L2 (spot) | REST API snapshots | Current | No |
| Order book L2 (perp) | REST API snapshots | Current | No |
| Order book L2 (historical) | Export API | Mar 2023 - present | Yes (cookies) |

**Coins:** BTC, ETH, BNB, SOL, XRP, AVAX, DOGE

**Architecture note:** The funding rate pipeline uses two data sources that are automatically merged:
- **CDN bulk download** -- daily ZIP files containing all instruments (Mar 2022 - ~Sept 2025)
- **REST API pagination** -- per-instrument backward pagination (~Dec 2025 - present)
- There is a ~3 month gap (Sept-Dec 2025) between the two sources; the CDN has not been updated to cover it.

---

## 0. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Ensure the pipeline root is on the path
PIPELINE_ROOT = Path.cwd().parent
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

# Pipeline modules
from src import config
from src.utils import setup_logger, ensure_dirs
from src.okx_client import OKXClient
from src.instrument_mapper import map_instruments
from src.chunking import StateManager
from src.funding_downloader import (
    download_all_funding_rates,
    download_funding_rates_cdn_bulk,
    download_funding_rate_rest,
    merge_funding_sources,
)
from src.orderbook_downloader import (
    download_all_orderbooks,
    collect_current_orderbook,
    download_orderbook_export,
)

import pandas as pd
import json

# Create all required directories
ensure_dirs()

print(f"Pipeline root : {config.PIPELINE_ROOT}")
print(f"Data directory: {config.DATA_DIR}")
print(f"Coins         : {config.COINS}")

## 1. Initialize Client & Map Instruments

Query the OKX public API to verify instrument IDs for each coin.

In [ ]:
# Initialize the OKX client (no auth needed for funding rates)
client = OKXClient()

# Map coins to their instrument IDs
instrument_map = map_instruments(client)

# Display the mapping
pd.DataFrame(instrument_map).T

## 2. Download Funding Rates

The full pipeline runs three phases automatically:

1. **CDN Bulk** -- downloads ~1,300 daily ZIP files from the OKX static CDN, each
   containing every instrument's funding rate for that day. Extracts only our 7 target
   coins. Takes ~22 min first run; completed months are skipped on re-run.

2. **REST API** -- paginates backward through `/api/v5/public/funding-rate-history`
   for each coin. Covers the most recent ~3 months. Takes ~5 seconds per coin.

3. **Merge** -- combines CDN + REST data, deduplicates, and writes one final CSV per
   instrument to `data/okx/raw/funding_rates/{COIN}_USDT_SWAP_funding.csv`.

In [ ]:
state = StateManager()

# This runs all three phases: CDN bulk -> REST API -> merge
funding_results = download_all_funding_rates(client, instrument_map, state=state)

In [ ]:
# Summary table
summary_rows = []
for coin, df in funding_results.items():
    if df.empty:
        summary_rows.append({'coin': coin, 'records': 0})
        continue
    ts = pd.to_datetime(df['timestamp'].astype(int), unit='ms')
    fr = df['funding_rate'].astype(float)
    diffs_h = df['timestamp'].astype(int).sort_values().diff().dropna() / 3_600_000
    summary_rows.append({
        'coin': coin,
        'records': len(df),
        'start': ts.min().strftime('%Y-%m-%d'),
        'end': ts.max().strftime('%Y-%m-%d'),
        'mean_rate_bps': f"{fr.mean() * 10_000:.2f}",
        'std_rate_bps': f"{fr.std() * 10_000:.2f}",
        'max_gap_hours': f"{diffs_h.max():.0f}",
    })

pd.DataFrame(summary_rows).set_index('coin')

## 3. Order Book Data

### 3a. Current Snapshots (no auth required)

Collect live L2 order book snapshots (depth=400) from the REST API.
These are point-in-time snapshots useful for bid-ask spread analysis
and as a starting point for ongoing data collection.

In [ ]:
# ── Spot order book snapshots ──
spot_rows = []
for coin in config.COINS:
    inst_id = instrument_map[coin]['spot']
    try:
        df = collect_current_orderbook(client, inst_id, 'SPOT', depth=400)
        asks = df[df['side'] == 'ask']
        bids = df[df['side'] == 'bid']
        spread_bps = (asks['price'].min() - bids['price'].max()) / bids['price'].max() * 10_000
        spot_rows.append({
            'coin': coin, 'type': 'SPOT',
            'best_bid': bids['price'].max(),
            'best_ask': asks['price'].min(),
            'spread_bps': f"{spread_bps:.2f}",
            'levels': len(asks),
        })
    except Exception as e:
        print(f"  {coin} SPOT failed: {e}")

pd.DataFrame(spot_rows).set_index('coin')

### 3b. Perpetual Order Book Snapshots

In [ ]:
# ── Perp order book snapshots ──
perp_rows = []
for coin in config.COINS:
    inst_id = instrument_map[coin]['perp']
    try:
        df = collect_current_orderbook(client, inst_id, 'SWAP', depth=400)
        asks = df[df['side'] == 'ask']
        bids = df[df['side'] == 'bid']
        spread_bps = (asks['price'].min() - bids['price'].max()) / bids['price'].max() * 10_000
        perp_rows.append({
            'coin': coin, 'type': 'PERP',
            'best_bid': bids['price'].max(),
            'best_ask': asks['price'].min(),
            'spread_bps': f"{spread_bps:.2f}",
            'levels': len(asks),
        })
    except Exception as e:
        print(f"  {coin} PERP failed: {e}")

pd.DataFrame(perp_rows).set_index('coin')

### 3c. Historical Order Book Export (requires OKX session cookies)

The OKX historical data export system at https://www.okx.com/en-us/historical-data
provides L2 snapshots from March 2023 onward via:

```
POST /priapi/v5/broker/public/trade-data/download-link
```

This endpoint requires browser session cookies. Without them it returns
`code: 0` but with empty `details`.

**To enable:**
1. Log in to OKX in your browser
2. Open DevTools (F12) -> Application -> Cookies for `www.okx.com`
3. Copy all cookie key-value pairs into the dict below
4. Uncomment and run the cell

In [ ]:
# ── Uncomment to enable historical order book export ──

# session_cookies = {
#     # Paste ALL cookies from your OKX browser session here.
#     "devId": "...",
#     "ok-sess": "...",
#     # ... add all cookie key-value pairs
# }
#
# auth_client = OKXClient(session_cookies=session_cookies)
#
# # Download historical order books for all coins (spot + perp)
# ob_results = download_all_orderbooks(
#     auth_client, instrument_map, state=state,
#     start_date="2023-03-01", method="export"
# )
# print("Results:", ob_results)

print("Historical order book export is disabled (no session cookies).")
print("Current snapshots were collected above.")

## 4. Data Verification

Check what files were downloaded and verify data integrity.

In [ ]:
# Full file listing
sections = [
    ('FUNDING RATES', config.FUNDING_DIR),
    ('SPOT ORDER BOOKS', config.SPOT_OB_DIR),
    ('PERP ORDER BOOKS', config.PERP_OB_DIR),
]

for title, directory in sections:
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")
    files = sorted(directory.glob('*.csv'))
    total_kb = 0
    for f in files:
        kb = f.stat().st_size / 1024
        total_kb += kb
        rows = sum(1 for _ in open(f)) - 1
        print(f"  {f.name:55s} {rows:>7,} rows  {kb:>7.1f} KB")
    print(f"  {'TOTAL':55s} {'':>7s}  {total_kb:>7.1f} KB")

In [ ]:
# Manifest / state
if config.MANIFEST_FILE.exists():
    with open(config.MANIFEST_FILE) as f:
        manifest = json.load(f)
    for key in sorted(manifest):
        v = manifest[key]
        status = 'COMPLETE' if v.get('complete') else 'partial'
        print(f"  {key:40s} [{status}]")
else:
    print("No manifest file yet.")

## 5. Sample Data Preview

In [ ]:
# Preview BTC funding rate data
btc_path = config.FUNDING_DIR / 'BTC_USDT_SWAP_funding.csv'
if btc_path.exists():
    df = pd.read_csv(btc_path)
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms')
    df['rate_bps'] = df['funding_rate'].astype(float) * 10_000
    print(f"BTC-USDT-SWAP Funding Rates")
    print(f"  Records : {len(df):,}")
    print(f"  Range   : {df['datetime'].min()} -> {df['datetime'].max()}")
    print(f"  Mean    : {df['rate_bps'].mean():.2f} bps")
    print()
    display(df[['datetime', 'funding_rate', 'realized_rate', 'instId']].head(10))
    display(df[['datetime', 'funding_rate', 'realized_rate', 'instId']].tail(10))
else:
    print('BTC funding data not found -- run Phase 2 first.')

In [ ]:
# Preview a spot order book snapshot
spot_files = sorted(config.SPOT_OB_DIR.glob('BTC_USDT_spot_*.csv'))
if spot_files:
    df = pd.read_csv(spot_files[-1])
    print(f"BTC-USDT Spot Order Book ({spot_files[-1].name})")
    print(f"  Asks: {len(df[df['side']=='ask'])} levels")
    print(f"  Bids: {len(df[df['side']=='bid'])} levels")
    display(df.head(5))
else:
    print('No BTC spot order book snapshot found.')